In [2]:
# =========================
# Q7 - Previsão de Demanda
# Produto: Motor de Popa Yamaha Evo Dash 155HP
# =========================

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# =========================
# 1. Carregar dados
# =========================

vendas = pd.read_csv("../data/raw/vendas_2023_2024.csv")
produtos = pd.read_csv("../data/processed/produtos_clean.csv")

# Converter datas
vendas["sale_date"] = pd.to_datetime(vendas["sale_date"], format="mixed", dayfirst=True)

# =========================
# 2. Selecionar produto alvo
# =========================

produto_nome = "Motor de Popa Yamaha Evo Dash 155HP"

produto_id = produtos.loc[
    produtos["name"] == produto_nome,
    "code"
].iloc[0]

print("Produto selecionado:", produto_nome)
print("ID:", produto_id)

# =========================
# 3. Filtrar vendas do produto
# =========================

df = vendas[vendas["id_product"] == produto_id].copy()

# =========================
# 4. Série temporal diária
# =========================

df_daily = (
    df.groupby("sale_date")["qtd"]
    .sum()
    .reset_index()
)

df_daily.columns = ["data", "qtd"]

# Criar calendário completo
date_range = pd.date_range(df_daily["data"].min(), df_daily["data"].max())

df_daily = (
    df_daily
    .set_index("data")
    .reindex(date_range, fill_value=0)
    .rename_axis("data")
    .reset_index()
)

# =========================
# 5. Separar treino e teste
# =========================

train = df_daily[df_daily["data"] <= "2023-12-31"].copy()
test = df_daily[
    (df_daily["data"] >= "2024-01-01") &
    (df_daily["data"] <= "2024-01-31")
].copy()

# =========================
# 6. Baseline: média móvel 7 dias
# =========================

historico = train["qtd"].tolist()
previsoes = []

for _ in range(len(test)):
    
    if len(historico) >= 7:
        pred = np.mean(historico[-7:])
    else:
        pred = np.mean(historico)

    previsoes.append(pred)

    historico.append(pred)

test["forecast"] = previsoes

# =========================
# 7. Avaliação do modelo
# =========================

mae = mean_absolute_error(test["qtd"], test["forecast"])

print("\nMAE:", round(mae, 2))

# =========================
# 8. Soma previsão 1ª semana
# =========================

primeira_semana = test[
    (test["data"] >= "2024-01-01") &
    (test["data"] <= "2024-01-07")
]

soma_prevista = round(primeira_semana["forecast"].sum())

print("\nSoma prevista (01–07 Jan):", soma_prevista)

# =========================
# 9. Resultado final
# =========================

test.head(10)

Produto selecionado: Motor de Popa Yamaha Evo Dash 155HP
ID: 54

MAE: 0.55

Soma prevista (01–07 Jan): 0


,data,qtd,forecast
356,2024-01-01,0,0.0
357,2024-01-02,0,0.0
358,2024-01-03,0,0.0
359,2024-01-04,0,0.0
360,2024-01-05,0,0.0
361,2024-01-06,0,0.0
362,2024-01-07,0,0.0
363,2024-01-08,0,0.0
364,2024-01-09,0,0.0
365,2024-01-10,0,0.0
